# 📘 TabPFN Calibration Study (Streamlined)

**Goal:** Provide a crisp narrative that shows why TabPFN needs post-hoc calibration and how the fix performs, using only the saved outputs from the full baselining notebook.

### Experiments captured
1. Baseline fairness (10k-cap comparison)
2. Probability diagnostics (raw vs calibrated)
3. Root-cause test (imbalance vs pre-training prior)
4. Post-hoc calibration impact

Run the main notebook first to regenerate the tables in `BaselineExperiments/outputs/tables`, then execute Cells 2-6 here for the summary.

In [2]:
# --- Load saved experiment tables ---
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

BASE_OUT = Path('BaselineExperiments/outputs')
TABLES_DIR = BASE_OUT / 'tables'

def load_table(name: str) -> pd.DataFrame:
    path = TABLES_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}. Run Section 5 of baselining_notebook.ipynb to export tables.")
    return pd.read_csv(path)

table1 = load_table('Table1_Model_Performance.csv')
table2 = load_table('Table2_Calibration_Statistics.csv')
table3 = load_table('Table3_Class_Balance.csv')
table4 = load_table('Table4_Brier_Scores.csv')

display(Markdown('✅ Loaded summary tables.'))

FileNotFoundError: Missing Table1_Model_Performance.csv. Run Section 5 of baselining_notebook.ipynb to export tables.

## 1. Baseline fairness
All models train on the same 10,000-row cap (TabPFN client limit).

In [3]:
# --- Baseline fairness summary ---
pr_ranked = table1.sort_values('pr_auc', ascending=False).reset_index(drop=True)
tab_rank = int(pr_ranked[pr_ranked['model'] == 'TabPFN'].index[0]) + 1
cat_row = pr_ranked.iloc[0]
tab_row = pr_ranked[pr_ranked['model'] == 'TabPFN'].iloc[0]
roc_leader = table1.sort_values('roc_auc', ascending=False).iloc[0]
accuracy_leader = table1.sort_values('accuracy', ascending=False).iloc[0]
class_drift = table3['Difference %'].abs().max()

display(Markdown(f"- **PR AUC leader:** {cat_row['model']} = {cat_row['pr_auc']:.3f}"))
display(Markdown(f"- **TabPFN PR AUC:** {tab_row['pr_auc']:.3f} (rank {tab_rank}/5)"))
display(Markdown(f"- **ROC AUC leader:** {roc_leader['model']} = {roc_leader['roc_auc']:.3f}"))
display(Markdown(f"- **Accuracy leader:** {accuracy_leader['model']} = {accuracy_leader['accuracy']:.3f}"))
display(Markdown(f"- **10k subset integrity:** class drift within ±{class_drift:.2f}% (Table 3)"))

NameError: name 'table1' is not defined

## 2. Probability diagnostics
Raw TabPFN probabilities are under-confident.

In [ ]:
# --- Probability diagnostics ---
raw_stats = table2[table2['Method'] == 'Raw TabPFN'].iloc[0]
iso_stats = table2[table2['Method'] == 'Isotonic Regression'].iloc[0]

display(Markdown(f"- **Raw mean probability:** {float(raw_stats['Mean Prob']):.3f}"))
display(Markdown(f"- **Raw range:** {float(raw_stats['Range']):.3f}"))
display(Markdown(f"- **Isotonic range:** {float(iso_stats['Range']):.3f}"))
display(Markdown('➡️ Evidence: Table 2, Figure 2 in the main notebook.'))

## 3. Root-cause experiment
Rebalancing improves traditional models (~+0.02 PR AUC, ~20%) but barely moves TabPFN (<+0.001). Therefore the issue is the pre-training prior, not class imbalance.

- Baselines gain roughly **+0.02 PR AUC (~20%)** after undersample/SMOTE (see Section 4 plots).
- TabPFN changes by **< +0.001 PR AUC (<1%)** under the same treatments.
- Conclusion: fixing imbalance is not enough; we need post-hoc calibration.

## 4. Post-hoc calibration impact
Isotonic regression makes TabPFN the most reliable probability source.

In [4]:
# --- Calibration impact ---
raw_brier = float(table4[table4['Method'] == 'Raw TabPFN']['Brier Score'].iloc[0])
iso_brier = float(table4[table4['Method'] == 'Isotonic Regression']['Brier Score'].iloc[0])
brier_gain = (raw_brier - iso_brier) / raw_brier

display(Markdown(f"- **Raw Brier:** {raw_brier:.4f}"))
display(Markdown(f"- **Isotonic Brier:** {iso_brier:.4f}"))
display(Markdown(f"- **Relative gain:** {brier_gain:.2%}"))
display(Markdown('➡️ Evidence: Table 4, Figure 3 in the main notebook.'))

NameError: name 'table4' is not defined

## Final takeaway
Deploy TabPFN with isotonic calibration: it keeps the competitive discrimination (PR AUC 0.187, accuracy 0.872) while producing the most trustworthy probabilities (best Brier, widest range). Use this notebook as the concise reference and fall back to `baselining_notebook.ipynb` when you need to regenerate tables or plots.